# IWTC Semantic Indexing

This notebook bootstraps the discovery and curation workflow for
entity relationships within a world repository.

It uses the canonical index tables produced by the raw source
indexing workflow to identify candidate entity pairs based on
co-occurrence evidence.

The notebook generates candidate relationship files for human
curation and processes the resulting canonical relationship index.

Artifacts are written to the repository's `working_drafts`
location for review before promotion to the canonical file:

- `world_relationships.csv`

This notebook operates on a single world repository configured by:

- `world_repository.yml`

# Pre-Build: Validate environment and load index artifacts

Validates repository paths and loads canonical index tables and vocabulary files into DataFrames.

No relationship candidates are generated here.

You may collapse this section after it runs successfully.

## Phase P0: Parameters

Define which world repository this notebook operates on and which index version it expects.

This notebook generates candidate relationship pairs from existing canonical index tables.

**IMPORTANT:** This notebook assumes index artifacts already exist and will fail if required CSV files are missing.

In [22]:
# Phase P0: Parameters
LAST_PHASE_RUN = "P0"

# Absolute path to the world_repository.yml descriptor.
WORLD_REPOSITORY_DESCRIPTOR = (
    "/Users/charissophia/obsidian/Iron Wolf Trading Company/_meta/descriptors/world_repository.yml"
)

# Index version to load (must match previously generated artifacts)
INDEX_VERSION = "V0"

# Internal run metadata (do not edit)
from datetime import datetime
print(f"Notebook run initialized at: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
del datetime

Notebook run initialized at: 2026-03-15 01:43


## Phase P1: Load and validate world descriptor

This phase verifies that the world repository descriptor is readable and structurally valid.

- Load the descriptor file
- Resolve required paths
- Confirm referenced directories exist

If validation fails, the notebook stops with actionable error messages.

No index tables are loaded until this succeeds.

In [82]:
# Phase P1: Load and validate world repository descriptor
LAST_PHASE_RUN = "P1"

from pathlib import Path
import yaml

errors = []
warnings = []

# --- Load descriptor file ---
descriptor_path = Path(WORLD_REPOSITORY_DESCRIPTOR)

if not descriptor_path.exists():
    raise FileNotFoundError(
        "World repository descriptor file was not found.\n"
        f"Path provided:\n  {descriptor_path}\n\n"
        "What to do:\n"
        "- Confirm the file exists at this location or fix WORLD_REPOSITORY_DESCRIPTOR in Phase 0\n"
        "- If you just edited Phase 0, rerun Phase 0 and then rerun this cell\n"
    )

try:
    with descriptor_path.open("r", encoding="utf-8") as f:
        world_repo = yaml.safe_load(f)
except Exception:
    raise ValueError(
        "The world repository descriptor could not be read.\n"
        "This usually indicates a YAML formatting problem.\n\n"
        f"File:\n  {descriptor_path}\n\n"
        "What to do:\n"
        "- Compare the file against the example world_repository.yml\n"
        "- Paste the contents into https://www.yamllint.com/\n"
        "- Fix any reported issues, save the file, and rerun this cell"
    )

if not isinstance(world_repo, dict):
    raise ValueError(
        "World repository descriptor structure is not usable.\n"
        "The file must be a YAML mapping (top-level `name: value` entries).\n"
    )

print(f"World repository descriptor loaded successfully: {descriptor_path.name}")

# --- Extract required entries ---
WORLD_ROOT_RAW = world_repo.get("world_root")

drafts_block = world_repo.get("working_drafts")
DRAFTS_RAW = drafts_block.get("path") if isinstance(drafts_block, dict) else None

indexes_block = world_repo.get("indexes")
INDEXES_RAW = indexes_block.get("path") if isinstance(indexes_block, dict) else None

vocab = world_repo.get("vocabulary") or {}
ENTITIES_RAW = vocab.get("entities")
RELATIONSHIPS_RAW = vocab.get("relationships")  # optional for now

if not WORLD_ROOT_RAW:
    errors.append("Missing required entry: world_root")
if not DRAFTS_RAW:
    errors.append("Missing required entry: working_drafts.path")
if not INDEXES_RAW:
    errors.append("Missing required entry: indexes.path")
if not ENTITIES_RAW:
    errors.append("Missing required entry: vocabulary.entities")

if errors:
    raise ValueError(
        "World repository descriptor is missing required entries:\n- "
        + "\n- ".join(errors)
        + "\n\nWhat to do:\n"
          "- Edit your world_repository.yml and add/fix the missing entries\n"
          "- Save the file and rerun this cell"
    )

# ------------------------------------------------------------------
# Published outputs (initialize up front for later phases)
# ------------------------------------------------------------------
WORLD_ROOT = None

WORKING_DRAFTS_PATH = None
WORKING_DRAFTS_RELPATH = None

INDEXES_PATH = None
INDEXES_RELPATH = None

VOCAB_ENTITIES_PATH = None
VOCAB_ENTITIES_RELPATH = None

VOCAB_RELATIONSHIPS_PATH = None
VOCAB_RELATIONSHIPS_RELPATH = None

# --- Validate and resolve world_root ---
WORLD_ROOT = Path(WORLD_ROOT_RAW)

if str(WORLD_ROOT).startswith("~"):
    errors.append("world_root: '~' is not allowed. Use a full absolute path.")
elif not WORLD_ROOT.is_absolute():
    errors.append("world_root must be an absolute path (starts with / on macOS/Linux, or C:\\ on Windows).")
elif not WORLD_ROOT.is_dir():
    errors.append(f"world_root must be an existing directory: {WORLD_ROOT}")
else:
    WORLD_ROOT = WORLD_ROOT.resolve()

if errors:
    raise ValueError("Descriptor path validation failed:\n- " + "\n- ".join(errors))

def _resolve_under_world_root(raw_path: str, label: str):
    if raw_path is None or str(raw_path).strip() == "":
        return None, None

    p = Path(str(raw_path))

    if str(p).startswith("~"):
        errors.append(f"{label}: '~' is not allowed: {raw_path}")
        return None, None

    if not p.is_absolute():
        p = WORLD_ROOT / p
    p = p.resolve()

    try:
        rel = str(p.relative_to(WORLD_ROOT))
    except Exception:
        rel = str(p)

    return p, rel

# --- Resolve and validate working_drafts path (required, directory, writable) ---
WORKING_DRAFTS_PATH, WORKING_DRAFTS_RELPATH = _resolve_under_world_root(DRAFTS_RAW, "working_drafts.path")

if WORKING_DRAFTS_PATH is None:
    errors.append("working_drafts.path: missing or invalid.")
else:
    if not WORKING_DRAFTS_PATH.exists():
        errors.append(f"working_drafts.path: path does not exist: {WORKING_DRAFTS_PATH}")
    elif not WORKING_DRAFTS_PATH.is_dir():
        errors.append(f"working_drafts.path: must be a directory: {WORKING_DRAFTS_PATH}")
    else:
        probe = WORKING_DRAFTS_PATH / ".iwtc_tools_write_probe.tmp"
        try:
            probe.write_text("test", encoding="utf-8")
        except Exception as e:
            errors.append(f"working_drafts.path: not writable: {WORKING_DRAFTS_PATH} ({type(e).__name__})")
        finally:
            try:
                if probe.exists():
                    probe.unlink()
            except Exception:
                pass
        del probe

# --- Resolve and validate indexes path (required, directory) ---
INDEXES_PATH, INDEXES_RELPATH = _resolve_under_world_root(INDEXES_RAW, "indexes.path")

if INDEXES_PATH is None:
    errors.append("indexes.path: missing or invalid.")
else:
    if not INDEXES_PATH.exists():
        errors.append(f"indexes.path: path does not exist: {INDEXES_PATH}")
    elif not INDEXES_PATH.is_dir():
        errors.append(f"indexes.path: must be a directory: {INDEXES_PATH}")

# --- Resolve and validate vocabulary.entities (required, file) ---
VOCAB_ENTITIES_PATH, VOCAB_ENTITIES_RELPATH = _resolve_under_world_root(ENTITIES_RAW, "vocabulary.entities")

if VOCAB_ENTITIES_PATH is None:
    errors.append("vocabulary.entities: missing or invalid.")
else:
    if not VOCAB_ENTITIES_PATH.exists():
        errors.append(f"vocabulary.entities: file does not exist: {VOCAB_ENTITIES_PATH}")
    elif VOCAB_ENTITIES_PATH.is_dir():
        errors.append(f"vocabulary.entities: must be a file, got directory: {VOCAB_ENTITIES_PATH}")

# --- Resolve vocabulary.relationships (optional for now) ---
if RELATIONSHIPS_RAW:
    VOCAB_RELATIONSHIPS_PATH, VOCAB_RELATIONSHIPS_RELPATH = _resolve_under_world_root(
        RELATIONSHIPS_RAW, "vocabulary.relationships"
    )

    if VOCAB_RELATIONSHIPS_PATH is not None:
        if VOCAB_RELATIONSHIPS_PATH.exists() and VOCAB_RELATIONSHIPS_PATH.is_dir():
            warnings.append(
                f"vocabulary.relationships: must be a file, got directory: {VOCAB_RELATIONSHIPS_PATH}"
            )
            VOCAB_RELATIONSHIPS_PATH = None
            VOCAB_RELATIONSHIPS_RELPATH = None
        elif not VOCAB_RELATIONSHIPS_PATH.exists():
            warnings.append(
                f"vocabulary.relationships: file does not yet exist: {VOCAB_RELATIONSHIPS_PATH}"
            )

if errors:
    raise ValueError("Descriptor path validation failed:\n- " + "\n- ".join(errors))

print("Descriptor paths are usable for this notebook.")
print(f"world_root: {WORLD_ROOT}")
print(f"working_drafts: {WORKING_DRAFTS_RELPATH}")
print(f"indexes: {INDEXES_RELPATH}")
print(
    f"vocabulary.entities: {VOCAB_ENTITIES_RELPATH} "
    f"(exists={VOCAB_ENTITIES_PATH.exists() if VOCAB_ENTITIES_PATH else False})"
)
print(
    f"vocabulary.relationships: {VOCAB_RELATIONSHIPS_RELPATH} "
    f"(exists={VOCAB_RELATIONSHIPS_PATH.exists() if VOCAB_RELATIONSHIPS_PATH else False})"
)

if warnings:
    print("\nWarnings:")
    for w in warnings:
        print(f"- {w}")

# cleanup
del yaml, Path
del descriptor_path, world_repo, drafts_block, indexes_block, vocab
del WORLD_ROOT_RAW, DRAFTS_RAW, INDEXES_RAW, ENTITIES_RAW, RELATIONSHIPS_RAW
del warnings, errors, f
del _resolve_under_world_root

World repository descriptor loaded successfully: world_repository.yml
Descriptor paths are usable for this notebook.
world_root: /Users/charissophia/obsidian/Iron Wolf Trading Company
working_drafts: _local/machine_wip
indexes: _meta/indexes
vocabulary.entities: _meta/indexes/vocab_entities.csv (exists=True)
vocabulary.relationships: _meta/indexes/world_relationships.csv (exists=True)


## Phase P2: Load required index and vocabulary artifacts

This phase verifies that the required generated index artifacts and required vocabulary files exist and can be loaded.

- Resolve expected versioned index filenames from `INDEX_VERSION`
- Load required generated index CSVs from `indexes.path`
- Load required vocabulary CSVs from descriptor-defined paths
- Verify required columns are present

If any artifact is missing or malformed, the notebook stops with instructions to correct the inputs.

No files are modified in this phase.

In [35]:
# Phase P2: Load required index and vocabulary artifacts
LAST_PHASE_RUN = "P2"

import pandas as pd
import itertools  # not used here, but needed during bootstrapping

errors = []

# Normalize INDEX_VERSION into the on-disk suffix
INDEX_VERSION_SUFFIX = f"v{str(INDEX_VERSION).lower().lstrip('v')}"

# Required generated index artifacts (versioned, loaded from indexes.path)
required_indexes = {
    "chunk_to_entities": f"index_chunk_to_entities_{INDEX_VERSION_SUFFIX}.csv",
    "source_files": f"index_source_files_{INDEX_VERSION_SUFFIX}.csv",
}

INDEX_FILES = {}

for key, fname in required_indexes.items():
    p = (INDEXES_PATH / fname).resolve()
    INDEX_FILES[key] = p
    if not p.exists():
        errors.append(f"Missing required artifact: {fname}\n  Expected at: {p}")

# Required vocabulary artifacts (descriptor-defined, not versioned)
if VOCAB_ENTITIES_PATH is None:
    errors.append("Missing required descriptor-resolved path: vocabulary.entities")
elif not VOCAB_ENTITIES_PATH.exists():
    errors.append(f"Missing required vocabulary file: {VOCAB_ENTITIES_PATH}")

if errors:
    raise FileNotFoundError(
        "Phase P2 cannot proceed because required artifacts are missing.\n\n"
        + "\n\n".join(errors)
        + "\n\nWhat to do:\n"
          "- Ensure raw source indexing has been executed\n"
          "- Confirm the required index CSV files exist under indexes.path\n"
          "- Confirm vocabulary.entities is correctly defined in world_repository.yml\n"
          "- Then rerun Phase P2"
    )

# Load CSVs
DF_CHUNK_TO_ENTITIES = pd.read_csv(INDEX_FILES["chunk_to_entities"])
DF_SOURCE_FILES = pd.read_csv(INDEX_FILES["source_files"])
DF_VOCAB_ENTITIES = pd.read_csv(VOCAB_ENTITIES_PATH)

# Validate minimal required columns
expected_cols = {
    "DF_CHUNK_TO_ENTITIES": {
        "chunk_id", "source_id", "source_type", "relpath",
        "chunk_start_line", "chunk_end_line",
        "entity_ids", "entity_count"
    },
    "DF_SOURCE_FILES": {"source_id", "relpath", "source_type"},
    "DF_VOCAB_ENTITIES": {"entity_id", "canonical_name", "entity_type", "layer"},
}

for df_name, cols in expected_cols.items():
    df = globals()[df_name]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        errors.append(f"{df_name}: missing expected columns: {missing}")

if errors:
    raise ValueError(
        "One or more artifacts were loaded but do not match expected structure.\n- "
        + "\n- ".join(errors)
        + "\n\nWhat to do:\n"
          "- Confirm you are using the expected canonical CSV artifacts\n"
          "- If necessary regenerate the index CSVs via IWTC_Raw_Source_Indexing.ipynb\n"
          "- Confirm vocabulary.entities points to the correct file in world_repository.yml"
    )

print("Phase P2 OK: required artifacts loaded.")
print(f"indexes.path: {INDEXES_PATH}")
print(f"index version: {INDEX_VERSION_SUFFIX}")

print("\nLoaded tables:")
print(f"- DF_CHUNK_TO_ENTITIES: {len(DF_CHUNK_TO_ENTITIES):>8} rows")
print(f"- DF_SOURCE_FILES:      {len(DF_SOURCE_FILES):>8} rows")
print(f"- DF_VOCAB_ENTITIES:    {len(DF_VOCAB_ENTITIES):>8} rows")

# cleanup
del errors, required_indexes, key, fname, p, df_name, df, cols, missing
del expected_cols, INDEX_FILES, INDEX_VERSION_SUFFIX

Phase P2 OK: required artifacts loaded.
indexes.path: /Users/charissophia/obsidian/Iron Wolf Trading Company/_meta/indexes
index version: v0

Loaded tables:
- DF_CHUNK_TO_ENTITIES:     1139 rows
- DF_SOURCE_FILES:           130 rows
- DF_VOCAB_ENTITIES:         178 rows


In [37]:
# --- Diagnostic: entity_type vs entity_id prefix distribution ---

DF_ENTITY_TYPE_PREFIX_SUMMARY = (
    DF_VOCAB_ENTITIES
    .assign(
        id_prefix=lambda df: df["entity_id"].str.split("_", n=1).str[0]
    )
    .groupby(["entity_type", "id_prefix"])
    .size()
    .reset_index(name="entity_count")
    .sort_values(["entity_type", "id_prefix"])
)

print("Entity type vs ID prefix distribution:\n")
display(DF_ENTITY_TYPE_PREFIX_SUMMARY)

del DF_ENTITY_TYPE_PREFIX_SUMMARY

Entity type vs ID prefix distribution:



,entity_type,id_prefix,entity_count
0,NPC,person,58
1,artifact,artifact,3
2,character,person,46
3,creature,creat,6
4,faction,faction,14
5,organization,org,11
6,place,place,27
7,player,player,12
8,reference,ref,1


In [83]:
# Phase P2 cleanup

# del INDEXES_PATH # need for G3
# del INDEXES_RELPATH # need for G1

del VOCAB_ENTITIES_PATH
# del VOCAB_ENTITIES_RELPATH # need for P7

# Relationship Bootstrapping

This section analyzes the indexed corpus to identify candidate entity
relationships based on co-occurrence evidence.

Entity pairs are discovered from chunk indexes, classified by pair
type, and aggregated into candidate relationship statistics used for
human curation.

## Phase R1: Normalize chunk entity evidence

This phase expands the chunk entity index into a normalized evidence
table with one row per `(chunk_id, entity_id)`.

The resulting table provides the foundation for relationship discovery
by making entity co-occurrence within chunks explicit.

No aggregation or candidate generation occurs in this phase.

In [38]:
# Phase R1: Normalize chunk entity evidence
LAST_PHASE_RUN = "R1"

# Create the new dataframe
DF_CHUNK_ENTITY_EVIDENCE = (
    DF_CHUNK_TO_ENTITIES[
        [
            "chunk_id",
            "source_id",
            "source_type",
            "relpath",
            "chunk_start_line",
            "chunk_end_line",
            "entity_ids",
        ]
    ]
    .assign(
        entity_id=lambda df: (
            df["entity_ids"]
            .fillna("")
            .astype(str)
            .str.split("|")
        )
    )
    .explode("entity_id", ignore_index=True)
    .assign(
        entity_id=lambda df: df["entity_id"].fillna("").astype(str).str.strip()
    )
    .loc[lambda df: df["entity_id"] != ""]
    .drop(columns=["entity_ids"])
    .drop_duplicates(subset=["chunk_id", "entity_id"])
    .reset_index(drop=True)
)

# Print out basic profile.
print("Phase R1 OK: chunk entity evidence normalized.")
print(f"- Input chunk rows:     {len(DF_CHUNK_TO_ENTITIES):>8}")
print(f"- Output evidence rows: {len(DF_CHUNK_ENTITY_EVIDENCE):>8}")
print(f"- Distinct chunks:      {DF_CHUNK_ENTITY_EVIDENCE['chunk_id'].nunique():>8}")
print(f"- Distinct entities:    {DF_CHUNK_ENTITY_EVIDENCE['entity_id'].nunique():>8}")

print("\nStructure:")
print(f"- rows: {len(DF_CHUNK_ENTITY_EVIDENCE)}")
print(f"- columns: {len(DF_CHUNK_ENTITY_EVIDENCE.columns)}")
print(f"- column names: {list(DF_CHUNK_ENTITY_EVIDENCE.columns)}")

print("\nSample rows:")
display(DF_CHUNK_ENTITY_EVIDENCE.head(3))



Phase R1 OK: chunk entity evidence normalized.
- Input chunk rows:         1139
- Output evidence rows:     5262
- Distinct chunks:          1139
- Distinct entities:         168

Structure:
- rows: 5262
- columns: 7
- column names: ['chunk_id', 'source_id', 'source_type', 'relpath', 'chunk_start_line', 'chunk_end_line', 'entity_id']

Sample rows:


,chunk_id,source_id,source_type,relpath,chunk_start_line,chunk_end_line,entity_id
0,168646,105,pbp_transcripts,_local/pbp_transcripts/PbP10 - The Second Camp.md,1,5,faction_spencer
1,168646,105,pbp_transcripts,_local/pbp_transcripts/PbP10 - The Second Camp.md,1,5,person_piers
2,168646,105,pbp_transcripts,_local/pbp_transcripts/PbP10 - The Second Camp.md,1,5,person_shworn


## Phase R2: Generate entity pairs within chunks

Generates all entity pairs observed within the same chunk.

Pairs are normalized by sorting entity IDs lexically so that each pair
appears only once per chunk.

Output columns:

- `chunk_id`
- `entity_id_a`
- `entity_id_b`
- `pair_type`

This phase captures co-occurrence evidence and derives pair typing
directly from entity ID prefixes.

In [8]:
# Phase R2: Generate entity pairs within chunks
LAST_PHASE_RUN = "R2"

# Build entity pairs per chunk
DF_CHUNK_ENTITY_PAIRS = (
    DF_CHUNK_ENTITY_EVIDENCE
    .groupby("chunk_id")["entity_id"]
    .apply(lambda entities: list(itertools.combinations(sorted(set(entities)), 2)))
    .explode()
    .dropna()
    .reset_index()
)

# Expand pairs and derive pair_type
DF_CHUNK_ENTITY_PAIRS = (
    DF_CHUNK_ENTITY_PAIRS
    .assign(
        entity_id_a=lambda df: df["entity_id"].str[0],
        entity_id_b=lambda df: df["entity_id"].str[1],
    )
    .drop(columns="entity_id")
    .assign(
        pair_type=lambda df: (
            df["entity_id_a"].str.split("_", n=1).str[0]
            + "|"
            + df["entity_id_b"].str.split("_", n=1).str[0]
        )
    )
)

# --- Diagnostic for R2 traceability ---
CHUNK_ENTITY_COUNTS = (
    DF_CHUNK_ENTITY_EVIDENCE
    .groupby("chunk_id")
    .size()
)

print("Phase R2 OK: entity pairs generated.")
print(f"- Input evidence rows:        {len(DF_CHUNK_ENTITY_EVIDENCE):>8}")
print(f"- Input chunks:               {CHUNK_ENTITY_COUNTS.shape[0]:>8}")
print(f"- Chunks with 0-1 entity:     {CHUNK_ENTITY_COUNTS.lt(2).sum():>8}")
print(f"- Chunks with 2+ entities:    {CHUNK_ENTITY_COUNTS.ge(2).sum():>8}")
print(f"- Output pair rows:           {len(DF_CHUNK_ENTITY_PAIRS):>8}")
print(f"- Output distinct chunks:     {DF_CHUNK_ENTITY_PAIRS['chunk_id'].nunique():>8}")
print(f"- Distinct pair types:        {DF_CHUNK_ENTITY_PAIRS['pair_type'].nunique():>8}")

print("\nStructure:")
print(f"- rows: {len(DF_CHUNK_ENTITY_PAIRS)}")
print(f"- columns: {len(DF_CHUNK_ENTITY_PAIRS.columns)}")
print(f"- column names: {list(DF_CHUNK_ENTITY_PAIRS.columns)}")

print("\nSample rows:")
display(DF_CHUNK_ENTITY_PAIRS.head(3))

del CHUNK_ENTITY_COUNTS

Phase R2 OK: entity pairs generated.
- Input evidence rows:            5262
- Input chunks:                   1139
- Chunks with 0-1 entity:          368
- Chunks with 2+ entities:         771
- Output pair rows:              26262
- Output distinct chunks:          771
- Distinct pair types:              35

Structure:
- rows: 26262
- columns: 4
- column names: ['chunk_id', 'entity_id_a', 'entity_id_b', 'pair_type']

Sample rows:


,chunk_id,entity_id_a,entity_id_b,pair_type
0,168646,faction_spencer,person_piers,faction|person
1,168646,faction_spencer,person_shworn,faction|person
2,168646,faction_spencer,person_victor,faction|person


## Phase R3: Analyze observed pair types

Summarizes the distribution of pair types observed in the corpus.

Pair types are derived from entity ID prefixes and represent the
ontology classes of entity relationships.

This phase is used to inspect which pair classes exist and how
frequently they occur, supporting decisions about which relationships
should be modeled and whether they should be directional or symmetric.

No transformations are applied to the pair table in this phase.

In [9]:
# Phase R3: Analyze observed pair types
LAST_PHASE_RUN = "R3"

DF_PAIR_TYPE_SUMMARY = (
    DF_CHUNK_ENTITY_PAIRS
    .groupby("pair_type")
    .size()
    .reset_index(name="pair_rows")
    .sort_values("pair_rows", ascending=False)
    .reset_index(drop=True)
)

print("Phase R3 OK: pair type distribution computed.")
print(f"- Distinct pair types: {len(DF_PAIR_TYPE_SUMMARY)}")

print("\nStructure:")
print(f"- rows: {len(DF_PAIR_TYPE_SUMMARY)}")
print(f"- columns: {len(DF_PAIR_TYPE_SUMMARY.columns)}")
print(f"- column names: {list(DF_PAIR_TYPE_SUMMARY.columns)}")

print("\nPair type distribution:")
display(DF_PAIR_TYPE_SUMMARY)

Phase R3 OK: pair type distribution computed.
- Distinct pair types: 35

Structure:
- rows: 35
- columns: 2
- column names: ['pair_type', 'pair_rows']

Pair type distribution:


,pair_type,pair_rows
0,person|person,11818
1,person|place,3400
2,org|person,2574
3,person|player,2048
4,faction|person,1551
5,creat|person,970
6,place|place,846
7,org|place,674
8,artifact|person,322
9,faction|place,295


In [10]:
# Phase R3.5: Generate editable pair type rule template
LAST_PHASE_RUN = "R3.5"

print("Copy the following cell and edit the rule values:\n")

print("DF_PAIR_TYPE_RULES = pd.DataFrame([")
for pair_type in DF_PAIR_TYPE_SUMMARY.sort_values("pair_type")["pair_type"]:
    print(
        f'    {{"pair_type": "{pair_type.ljust(DF_PAIR_TYPE_SUMMARY["pair_type"].str.len().max())}", '
        f'"include": True,  "directional": False}},'
    )
print("])")

Copy the following cell and edit the rule values:

DF_PAIR_TYPE_RULES = pd.DataFrame([
    {"pair_type": "artifact|artifact", "include": True,  "directional": False},
    {"pair_type": "artifact|creat   ", "include": True,  "directional": False},
    {"pair_type": "artifact|faction ", "include": True,  "directional": False},
    {"pair_type": "artifact|org     ", "include": True,  "directional": False},
    {"pair_type": "artifact|person  ", "include": True,  "directional": False},
    {"pair_type": "artifact|place   ", "include": True,  "directional": False},
    {"pair_type": "artifact|player  ", "include": True,  "directional": False},
    {"pair_type": "artifact|ref     ", "include": True,  "directional": False},
    {"pair_type": "creat|creat      ", "include": True,  "directional": False},
    {"pair_type": "creat|faction    ", "include": True,  "directional": False},
    {"pair_type": "creat|org        ", "include": True,  "directional": False},
    {"pair_type": "creat|person  

In [70]:
# R3.5 entry
# Remember this is first pass. Omit the noisiest that can be inferred by other relationships.
DF_PAIR_TYPE_RULES = pd.DataFrame([
    {"pair_type": "artifact|artifact", "include": False,  "directional": False},
    {"pair_type": "artifact|creat   ", "include": False,  "directional": False},
    {"pair_type": "artifact|faction ", "include": False,  "directional": True},
    {"pair_type": "artifact|org     ", "include": False,  "directional": True},
    {"pair_type": "artifact|person  ", "include": False,  "directional": False},
    {"pair_type": "artifact|place   ", "include": False,  "directional": True},
    {"pair_type": "artifact|player  ", "include": False,  "directional": False},
    {"pair_type": "artifact|ref     ", "include": False,  "directional": False},
    {"pair_type": "creat|creat      ", "include": False,  "directional": False},
    {"pair_type": "creat|faction    ", "include": False,  "directional": False},
    {"pair_type": "creat|org        ", "include": False,  "directional": False},
    {"pair_type": "creat|person     ", "include": False,  "directional": False},
    {"pair_type": "creat|place      ", "include": False,  "directional": False},
    {"pair_type": "creat|player     ", "include": False,  "directional": False},
    {"pair_type": "creat|ref        ", "include": False,  "directional": False},
    {"pair_type": "faction|faction  ", "include": True,  "directional": False},
    {"pair_type": "faction|org      ", "include": False,  "directional": True},
    {"pair_type": "faction|person   ", "include": False,  "directional": True},
    {"pair_type": "faction|place    ", "include": False,  "directional": True},
    {"pair_type": "faction|player   ", "include": False,  "directional": False},
    {"pair_type": "faction|ref      ", "include": False,  "directional": False},
    {"pair_type": "org|org          ", "include": False,  "directional": False},
    {"pair_type": "org|person       ", "include": False,  "directional": True},
    {"pair_type": "org|place        ", "include": False,  "directional": True},
    {"pair_type": "org|player       ", "include": False,  "directional": False},
    {"pair_type": "org|ref          ", "include": False,  "directional": False},
    {"pair_type": "person|person    ", "include": False,  "directional": False},
    {"pair_type": "person|place     ", "include": False,  "directional": True},
    {"pair_type": "person|player    ", "include": False,  "directional": True},
    {"pair_type": "person|ref       ", "include": False,  "directional": False},
    {"pair_type": "place|place      ", "include": False,  "directional": False},
    {"pair_type": "place|player     ", "include": False,  "directional": False},
    {"pair_type": "place|ref        ", "include": False,  "directional": False},
    {"pair_type": "player|player    ", "include": False,  "directional": False},
    {"pair_type": "player|ref       ", "include": False,  "directional": False},
])
DF_PAIR_TYPE_RULES["pair_type"] = DF_PAIR_TYPE_RULES["pair_type"].str.strip()

## Phase R4: Generate relationship candidates

This phase applies the reviewed pair type policy and aggregates chunk-level
pair evidence into candidate relationship rows for human curation.

The output remains evidence-backed and non-canonical:

- only pair types marked for inclusion are retained
- pair counts are aggregated across chunks
- source file counts and source type summaries are preserved
- `predicate` and `curation_notes` are left blank for human review

Directional pair types are flagged, but subject/object ordering remains
lexical in this phase.

In [36]:
# Phase R4: Generate relationship candidates
LAST_PHASE_RUN = "R4"

# Apply pair type rules to the chunk-level pair evidence
DF_RELATIONSHIP_CANDIDATES = (
    DF_CHUNK_ENTITY_PAIRS
    .merge(DF_PAIR_TYPE_RULES, on="pair_type", how="inner")
    .loc[lambda df: df["include"]]
    .merge(
        DF_CHUNK_ENTITY_EVIDENCE[
            ["chunk_id", "source_id", "source_type", "relpath"]
        ].drop_duplicates(),
        on="chunk_id",
        how="left"
    )
    .groupby(
        ["entity_id_a", "entity_id_b", "pair_type", "directional"],
        as_index=False
    )
    .agg(
        shared_chunk_count=("chunk_id", "nunique"),
        shared_file_count=("relpath", "nunique"),
        source_types=("source_type", lambda s: "|".join(sorted(set(s.dropna())))),
    )
    .rename(
        columns={
            "entity_id_a": "subject_id",
            "entity_id_b": "object_id",
        }
    )
    .assign(
        predicate="",
        curation_notes="",
        candidate_score=lambda df: df["shared_chunk_count"]
    )
    [
        [
            "subject_id",
            "predicate",
            "object_id",
            "pair_type",
            "directional",
            "shared_chunk_count",
            "shared_file_count",
            "source_types",
            "candidate_score",
            "curation_notes",
        ]
    ]
    .sort_values(
        ["candidate_score", "shared_file_count", "pair_type", "subject_id", "object_id"],
        ascending=[False, False, True, True, True]
    )
    .reset_index(drop=True)
)

print("Phase R4 OK: relationship candidates generated.")
print(f"- Input pair rows:              {len(DF_CHUNK_ENTITY_PAIRS):>8}")
print(f"- Included pair types:          {DF_PAIR_TYPE_RULES['include'].sum():>8}")
print(f"- Output candidate rows:        {len(DF_RELATIONSHIP_CANDIDATES):>8}")
print(f"- Distinct candidate pair types:{DF_RELATIONSHIP_CANDIDATES['pair_type'].nunique():>8}")

print("\nStructure:")
print(f"- rows: {len(DF_RELATIONSHIP_CANDIDATES)}")
print(f"- columns: {len(DF_RELATIONSHIP_CANDIDATES.columns)}")
print(f"- column names: {list(DF_RELATIONSHIP_CANDIDATES.columns)}")

print("\nSummary:")
display(
    DF_RELATIONSHIP_CANDIDATES["pair_type"]
    .value_counts()
    .rename_axis("pair_type")
    .reset_index(name="candidate_rows")
)

print("\nSample rows:")
display(DF_RELATIONSHIP_CANDIDATES.head(10))

Phase R4 OK: relationship candidates generated.
- Input pair rows:                 26262
- Included pair types:                 1
- Output candidate rows:              25
- Distinct candidate pair types:       1

Structure:
- rows: 25
- columns: 10
- column names: ['subject_id', 'predicate', 'object_id', 'pair_type', 'directional', 'shared_chunk_count', 'shared_file_count', 'source_types', 'candidate_score', 'curation_notes']

Summary:


,pair_type,candidate_rows
0,faction|faction,25



Sample rows:


,subject_id,predicate,object_id,pair_type,directional,shared_chunk_count,shared_file_count,source_types,candidate_score,curation_notes
0,faction_remedy,,faction_tolanites,faction|faction,False,18,9,pbp_transcripts|planning_notes|session_notes,18,
1,faction_hands,,faction_remedy,faction|faction,False,12,7,pbp_transcripts|planning_notes,12,
2,faction_hands,,faction_tolanites,faction|faction,False,12,7,pbp_transcripts|session_notes,12,
3,faction_hands,,faction_sleepsong,faction|faction,False,6,4,pbp_transcripts|session_notes,6,
4,faction_hands,,faction_thieves_guild,faction|faction,False,6,2,pbp_transcripts,6,
5,faction_halverin,,faction_hands,faction|faction,False,5,4,pbp_transcripts|planning_notes,5,
6,faction_michaeline,,faction_remedy,faction|faction,False,4,2,pbp_transcripts|session_notes,4,
7,faction_foreclaimers,,faction_hands,faction|faction,False,3,2,session_notes,3,
8,faction_remedy,,faction_spencer,faction|faction,False,3,2,planning_notes|session_notes,3,
9,faction_spencer,,faction_tolanites,faction|faction,False,3,2,planning_notes|session_notes,3,


## Phase R5: Export Relationship Candidates for Curation

This phase writes the generated relationship candidates to a CSV file for human review and curation.

The candidate table represents **evidence-backed relationship suggestions** derived from entity co-occurrence in indexed chunks. It is not yet canonical world data. Instead, it provides a structured starting point for identifying meaningful relationships.

Each candidate includes:

- `subject_id` and `object_id` – the entity pair
- `pair_type` – semantic class derived from entity prefixes
- `directional` – whether the relationship is likely directional
- `shared_chunk_count` – number of chunks where the pair co-occurs
- `shared_file_count` – number of files containing those chunks
- `source_types` – summary of source categories contributing evidence
- `candidate_score` – heuristic ranking value (currently chunk count)
- `predicate` – blank column for human curation
- `curation_notes` – optional field for reviewer comments

The exported file is intended to be reviewed outside the notebook (e.g., spreadsheet editor). During curation, reviewers will:

1. Assign a `predicate` where a meaningful relationship exists.
2. Leave rows blank where the co-occurrence is incidental.
3. Add notes if interpretation is uncertain.

A curated version of this dataset can later be promoted to the repository as the canonical relationship table (e.g., `world_relationships.csv`).

In [37]:
# Phase R5: Export relationship candidates
LAST_PHASE_RUN = "R5"

from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_PATH = (
    WORKING_DRAFTS_PATH
    / f"candidate_relationships_{INDEX_VERSION.lower()}_{timestamp}.csv"
)

(
    DF_RELATIONSHIP_CANDIDATES
    .sort_values(
        ["pair_type", "candidate_score", "subject_id", "object_id"],
        ascending=[True, False, True, True],
    )
    .to_csv(OUTPUT_PATH, index=False)
)

print("Phase R5 OK: candidate relationships written.")
print(f"- Output path: {OUTPUT_PATH}")
print(f"- Rows written: {len(DF_RELATIONSHIP_CANDIDATES)}")
print(f"- Columns: {len(DF_RELATIONSHIP_CANDIDATES.columns)}")

del timestamp, OUTPUT_PATH, Path, datetime

Phase R5 OK: candidate relationships written.
- Output path: /Users/charissophia/obsidian/Iron Wolf Trading Company/_local/machine_wip/candidate_relationships_v0_20260306_173518.csv
- Rows written: 25
- Columns: 10


In [30]:
# example weak candidates
display(
    DF_RELATIONSHIP_CANDIDATES
    .sort_values(
        ["pair_type", "shared_chunk_count"],
        ascending=[True, True]
    )
    .groupby("pair_type")
    .head(10)
    .reset_index(drop=True)
)

,subject_id,predicate,object_id,pair_type,directional,shared_chunk_count,shared_file_count,source_types,candidate_score,curation_notes
0,artifact_iris,,faction_modrons,artifact|faction,True,2,1,session_notes,2,
1,artifact_folly,,faction_hands,artifact|faction,True,5,3,pbp_transcripts|session_notes,5,
2,artifact_iris,,org_myconids,artifact|org,True,2,1,session_notes,2,
3,artifact_folly,,org_myconids,artifact|org,True,3,1,session_notes,3,
4,artifact_folly,,org_king,artifact|org,True,4,2,pbp_transcripts|session_notes,4,
...,...,...,...,...,...,...,...,...,...,...
63,person_bre,,place_druwen,person|place,True,2,2,session_notes,2,
64,person_bre,,place_elulind,person|place,True,2,2,planning_notes|session_notes,2,
65,person_bre,,place_wolfstream,person|place,True,2,2,planning_notes|session_notes,2,
66,person_brigitte,,place_northbend,person|place,True,2,2,planning_notes|session_notes,2,


In [75]:
# Phase R5A: Export entity type compatibility policy
# (generated from observed pair types)

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_PATH = (
    WORKING_DRAFTS_PATH
    / f"data_policy_entity_type_compatibility_{timestamp}.csv"
)

(
    DF_PAIR_TYPE_RULES
    .sort_values(["pair_type"])
    .to_csv(OUTPUT_PATH, index=False)
)

print("Phase R5A OK: entity type compatibility policy written.")
print(f"- Output path: {OUTPUT_PATH}")
print(f"- Rows written: {len(DF_PAIR_TYPE_RULES)}")
print(f"- Columns: {len(DF_PAIR_TYPE_RULES.columns)}")

print("\nNext step (human review required):")
print("1. Open the generated CSV file.")
print("2. Review each pair_type and confirm the compatibility settings.")
print("3. Adjust any values if necessary.")
print("4. When satisfied, move the file to:")
print(f"   {INDEXES_RELPATH}/data_policy_entity_type_compatibility.csv")
print("5. Remove the timestamp from the filename when promoting it.")

del timestamp, OUTPUT_PATH, datetime

Phase R5A OK: entity type compatibility policy written.
- Output path: /Users/charissophia/obsidian/Iron Wolf Trading Company/_local/machine_wip/data_policy_entity_type_compatibility_20260315_042004.csv
- Rows written: 35
- Columns: 3

Next step (human review required):
1. Open the generated CSV file.
2. Review each pair_type and confirm the compatibility settings.
3. Adjust any values if necessary.
4. When satisfied, move the file to:
   _meta/indexes/data_policy_entity_type_compatibility.csv
5. Remove the timestamp from the filename when promoting it.


## Phase R6: Load canonical relationships

Loads the curated canonical relationship table from the descriptor-defined
vocabulary path and normalizes its schema to stable semantic column names.

This phase:

- loads the canonical relationship CSV
- accepts alternate human-facing column titles
- normalizes to `subject_id`, `predicate`, and `object_id`
- strips whitespace and removes empty rows
- prepares the relationship table for downstream semantic use

No graph artifacts are built in this phase.

In [59]:
# Phase R6: Load canonical relationships
# Output:
#   - DF_RELATIONSHIPS
#
# Notes:
# - CSV schema is human-facing and may contain extra columns.
# - We normalize to semantic column names via RELATIONSHIP_COLS below.
# - Any columns not mapped are intentionally ignored.
LAST_PHASE_RUN = "R6"

from pathlib import Path

# ------------------------------------------------------------------
# Semantic column mappings for canonical relationship CSV
# Each semantic field -> acceptable column names in the CSV
# Any other CSV columns are ignored on purpose.
# ------------------------------------------------------------------
RELATIONSHIP_COLS = {
    "subject_id": ["subject_id", "subject", "source_id", "from_id"],
    "predicate":  ["predicate", "relationship", "relation", "edge", "verb"],
    "object_id":  ["object_id", "object", "target_id", "to_id"],
}

# ------------------------------------------------------------------
# Helper function to handle CSV column names
# ------------------------------------------------------------------
def _normalize_relationship_csv(df, col_map, label):

    if df is None or df.empty:
        return pd.DataFrame(columns=list(col_map.keys()))

    rename = {}
    for semantic, options in col_map.items():
        found = next((c for c in options if c in df.columns), None)
        if found:
            rename[found] = semantic

    if len(df) > 0 and not rename:
        header_line = ", ".join(list(df.columns))
        expected = {k: v for k, v in col_map.items()}
        print(
            f"WARNING [{label}]: CSV has rows but none of the expected columns were found.\n"
            f"  CSV columns: {header_line}\n"
            f"  Expected (semantic -> acceptable names): {expected}"
        )
        return pd.DataFrame(columns=list(col_map.keys()))

    out = df.rename(columns=rename)

    missing_semantic = [k for k in col_map.keys() if k not in out.columns]
    if len(df) > 0 and missing_semantic:
        print(
            f"WARNING [{label}]: missing semantic columns after normalization: {missing_semantic}\n"
            f"  CSV columns: {list(df.columns)}\n"
            f"  Using: {list(out.columns)}"
        )

    keep = [k for k in col_map.keys() if k in out.columns]
    return out[keep].copy()

# ------------------------------------------------------------------
# Validate required globals
# ------------------------------------------------------------------
if "VOCAB_RELATIONSHIPS_PATH" not in globals() or not VOCAB_RELATIONSHIPS_PATH:
    raise ValueError(
        "VOCAB_RELATIONSHIPS_PATH is missing or empty.\n"
        "Rerun Phase P1 after adding vocabulary.relationships to the descriptor."
    )

relationships_rel = globals().get("VOCAB_RELATIONSHIPS_RELPATH", "none")

# ------------------------------------------------------------------
# Load canonical relationships (required) + normalize schema
# ------------------------------------------------------------------
relationships_path = Path(VOCAB_RELATIONSHIPS_PATH)
if not relationships_path.exists():
    raise ValueError(f"Canonical relationship file not found: {relationships_path}")

relationships_raw = pd.read_csv(relationships_path, dtype=str).fillna("")
DF_RELATIONSHIPS = _normalize_relationship_csv(
    relationships_raw,
    RELATIONSHIP_COLS,
    "relationships"
)

if DF_RELATIONSHIPS.empty:
    raise ValueError("Relationship CSV did not produce any usable rows after normalization.")

# ------------------------------------------------------------------
# Normalize whitespace and drop empty rows
# ------------------------------------------------------------------
before_cnt = len(DF_RELATIONSHIPS)

DF_RELATIONSHIPS = (
    DF_RELATIONSHIPS
    .assign(
        subject_id=lambda df: df["subject_id"].astype(str).str.strip(),
        predicate=lambda df: df["predicate"].astype(str).str.strip(),
        object_id=lambda df: df["object_id"].astype(str).str.strip(),
    )
    .loc[lambda df: (df["subject_id"] != "") & (df["predicate"] != "") & (df["object_id"] != "")]
    .drop_duplicates()
    .reset_index(drop=True)
)

# ------------------------------------------------------------------
# Profile loaded relationships
# ------------------------------------------------------------------
print(f"Loaded canonical relationships: {before_cnt} [{relationships_rel}]")
print(f"Rows after cleanup: {len(DF_RELATIONSHIPS)} (removed {before_cnt - len(DF_RELATIONSHIPS)})")
print(f"Distinct predicates: {DF_RELATIONSHIPS['predicate'].nunique()}")

print("\nStructure:")
print(f"- rows: {len(DF_RELATIONSHIPS)}")
print(f"- columns: {len(DF_RELATIONSHIPS.columns)}")
print(f"- column names: {list(DF_RELATIONSHIPS.columns)}")

print("\nPredicate distribution:")
display(
    DF_RELATIONSHIPS["predicate"]
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="relationship_count")
)

print("\nSample rows:")
display(DF_RELATIONSHIPS.head(10))

# Cleanup locals (keep DF_RELATIONSHIPS)
del relationships_path, relationships_raw, relationships_rel
del RELATIONSHIP_COLS, before_cnt

Loaded canonical relationships: 148 [_meta/indexes/world_relationships.csv]
Rows after cleanup: 144 (removed 4)
Distinct predicates: 24

Structure:
- rows: 144
- columns: 3
- column names: ['subject_id', 'predicate', 'object_id']

Predicate distribution:


,predicate,relationship_count
0,current_member_of,44
1,former_member_of,24
2,based_at,16
3,allied_with,8
4,belongs_to,8
5,opposes,8
6,leader_of,7
7,neighbor_with,5
8,conquered,3
9,rescued,3



Sample rows:


,subject_id,predicate,object_id
0,faction_hands,allied_with,faction_thieves_guild
1,faction_michaeline,allied_with,faction_remedy
2,faction_remedy,allied_with,faction_spencer
3,faction_remedy,allied_with,faction_tolanites
4,faction_sleepsong,allied_with,person_kir
5,faction_thieves_guild,allied_with,person_victor
6,org_iwtc,allied_with,place_stonehome
7,org_ravencrest,allied_with,person_avarna
8,person_jane,ambassador_to,org_myconids
9,org_king,at_war_with,place_tolan


## Phase R7: Validate canonical relationships against entity vocabulary

This phase verifies that all relationship endpoints reference known
entities from the vocabulary.

Checks performed:

- subject_id must exist in DF_VOCAB_ENTITIES
- object_id must exist in DF_VOCAB_ENTITIES
- reports unknown IDs if present

This prevents typos or stale entity references from entering the
relationship graph.

In [60]:
# Phase R7: Validate canonical relationships against entity vocabulary
LAST_PHASE_RUN = "R7"

KNOWN_ENTITY_IDS = set(DF_VOCAB_ENTITIES["entity_id"])

unknown_subjects = (
    DF_RELATIONSHIPS.loc[~DF_RELATIONSHIPS["subject_id"].isin(KNOWN_ENTITY_IDS), "subject_id"]
    .drop_duplicates()
    .sort_values()
)

unknown_objects = (
    DF_RELATIONSHIPS.loc[~DF_RELATIONSHIPS["object_id"].isin(KNOWN_ENTITY_IDS), "object_id"]
    .drop_duplicates()
    .sort_values()
)

if len(unknown_subjects) or len(unknown_objects):

    print("Unknown subject_ids:")
    display(unknown_subjects)

    print("\nUnknown object_ids:")
    display(unknown_objects)

    raise ValueError(
        "Canonical relationships reference unknown entity IDs.\n\n"
        "What to do:\n"
        f"- If the relationship file is wrong, update {VOCAB_RELATIONSHIPS_RELPATH},\n"
        "  rerun Phase R6, then rerun this phase.\n"
        "- If the relationship is correct but the entity is missing from the vocabulary,\n"
        f"  update {VOCAB_ENTITIES_RELPATH} and rerun the Pre-build section, then\n"
        "  rerun this phase."
    )

print("Phase R7 OK: all relationship endpoints reference known entities.")
print(f"- Relationships checked: {len(DF_RELATIONSHIPS)}")
print(f"- Distinct predicates:   {DF_RELATIONSHIPS['predicate'].nunique()}")

print("\nPredicate distribution:")
display(
    DF_RELATIONSHIPS["predicate"]
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="relationship_count")
)

del KNOWN_ENTITY_IDS, unknown_subjects, unknown_objects

Phase R7 OK: all relationship endpoints reference known entities.
- Relationships checked: 144
- Distinct predicates:   24

Predicate distribution:


,predicate,relationship_count
0,current_member_of,44
1,former_member_of,24
2,based_at,16
3,allied_with,8
4,belongs_to,8
5,opposes,8
6,leader_of,7
7,neighbor_with,5
8,conquered,3
9,rescued,3


In [61]:
# optional profiling
display(
    DF_RELATIONSHIPS
    .assign(
        subject_type=lambda df: df["subject_id"].str.split("_", n=1).str[0],
        object_type=lambda df: df["object_id"].str.split("_", n=1).str[0],
    )
    .groupby(["subject_type", "predicate", "object_type"])
    .size()
    .reset_index(name="relationship_count")
    .sort_values(
        ["relationship_count", "predicate", "subject_type", "object_type"],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)

,subject_type,predicate,object_type,relationship_count
0,person,current_member_of,org,30
1,person,former_member_of,org,21
2,person,current_member_of,faction,10
3,faction,based_at,place,7
4,org,based_at,place,7
5,creat,belongs_to,person,6
6,faction,neighbor_with,faction,5
7,faction,allied_with,faction,4
8,person,leader_of,faction,4
9,faction,opposes,org,4


In [58]:
# optional test of a predicate
TEST_PREDICATE = "current_member_of"

TEST_REL = (
    DF_RELATIONSHIPS
    .query("predicate == @TEST_PREDICATE")
    .assign(
        subject_type=lambda df: df["subject_id"].str.split("_", n=1).str[0],
        object_type=lambda df: df["object_id"].str.split("_", n=1).str[0],
    )
)

display(
    TEST_REL[["subject_id","predicate","object_id"]]
)

display(
    TEST_REL
    .groupby(["subject_type","predicate","object_type"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

# clean up locals
del TEST_PREDICATE, TEST_REL

,subject_id,predicate,object_id
38,person_alivyre,current_member_of,org_party
39,person_aniya,current_member_of,org_party
40,person_cara,current_member_of,org_party
41,person_credus,current_member_of,org_party
42,person_cunai,current_member_of,org_party
43,person_elaris,current_member_of,org_party
44,person_faeryne,current_member_of,org_party
45,person_holly,current_member_of,org_party
46,person_katy,current_member_of,org_party
47,person_liacyne,current_member_of,org_party


,subject_type,predicate,object_type,count
3,person,current_member_of,org,30
2,person,current_member_of,faction,10
1,faction,current_member_of,org,3
0,faction,current_member_of,faction,1


## Phase R8: Bootstrap predicate candidates

This phase summarizes the predicates currently present in the canonical
relationship table and generates a starter policy table for human review.

The policy table is intended to support later classification of predicates,
such as:

- whether a predicate should be included in the semantic graph
- whether it is symmetric
- what semantic class it belongs to

No graph artifacts are built in this phase.

In [63]:
# Phase R8: Bootstrap predicate candidates
LAST_PHASE_RUN = "R8"

DF_PREDICATE_SUMMARY = (
    DF_RELATIONSHIPS["predicate"]
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="relationship_count")
    .sort_values("predicate")
    .reset_index(drop=True)
)

print("Phase R8 OK: predicate candidates summarized.")
print(f"- Distinct predicates: {len(DF_PREDICATE_SUMMARY)}")

print("\nStructure:")
print(f"- rows: {len(DF_PREDICATE_SUMMARY)}")
print(f"- columns: {len(DF_PREDICATE_SUMMARY.columns)}")
print(f"- column names: {list(DF_PREDICATE_SUMMARY.columns)}")

print("\nPredicate summary:")
display(DF_PREDICATE_SUMMARY)

print("\nCopy the following cell and edit the rule values:\n")

print("DF_PREDICATE_RULES = pd.DataFrame([")
for predicate in DF_PREDICATE_SUMMARY["predicate"]:
    print(
        f'    {{"predicate": "{predicate.ljust(DF_PREDICATE_SUMMARY["predicate"].str.len().max())}", '
        f'"include": True, '
        f'"symmetric": False, '
        f'"relationship_class": ""}},'
    )
print("])")

Phase R8 OK: predicate candidates summarized.
- Distinct predicates: 24

Structure:
- rows: 24
- columns: 2
- column names: ['predicate', 'relationship_count']

Predicate summary:


,predicate,relationship_count
0,allied_with,8
1,ambassador_to,1
2,at_war_with,1
3,based_at,16
4,belongs_to,8
5,built,1
6,conquered,3
7,current_member_of,44
8,defender_of,1
9,former_member_of,24



Copy the following cell and edit the rule values:

DF_PREDICATE_RULES = pd.DataFrame([
    {"predicate": "allied_with        ", "include": True, "symmetric": False, "relationship_class": ""},
    {"predicate": "ambassador_to      ", "include": True, "symmetric": False, "relationship_class": ""},
    {"predicate": "at_war_with        ", "include": True, "symmetric": False, "relationship_class": ""},
    {"predicate": "based_at           ", "include": True, "symmetric": False, "relationship_class": ""},
    {"predicate": "belongs_to         ", "include": True, "symmetric": False, "relationship_class": ""},
    {"predicate": "built              ", "include": True, "symmetric": False, "relationship_class": ""},
    {"predicate": "conquered          ", "include": True, "symmetric": False, "relationship_class": ""},
    {"predicate": "current_member_of  ", "include": True, "symmetric": False, "relationship_class": ""},
    {"predicate": "defender_of        ", "include": True, "symmetric": F

In [64]:
# R8.5 entry
DF_PREDICATE_RULES = pd.DataFrame([
    {"predicate": "allied_with        ", "include": True, "symmetric": True,  "relationship_class": "political"},
    {"predicate": "ambassador_to      ", "include": True, "symmetric": False, "relationship_class": "diplomatic"},
    {"predicate": "at_war_with        ", "include": True, "symmetric": True,  "relationship_class": "political"},
    {"predicate": "based_at           ", "include": True, "symmetric": False, "relationship_class": "location"},
    {"predicate": "belongs_to         ", "include": True, "symmetric": False, "relationship_class": "ownership"},
    {"predicate": "built              ", "include": True, "symmetric": False, "relationship_class": "event"},
    {"predicate": "conquered          ", "include": True, "symmetric": False, "relationship_class": "event"},
    {"predicate": "current_member_of  ", "include": True, "symmetric": False, "relationship_class": "membership"},
    {"predicate": "defender_of        ", "include": True, "symmetric": False, "relationship_class": "political"},
    {"predicate": "former_member_of   ", "include": True, "symmetric": False, "relationship_class": "membership"},
    {"predicate": "invaded            ", "include": True, "symmetric": False, "relationship_class": "event"},
    {"predicate": "investigating      ", "include": True, "symmetric": False, "relationship_class": "activity"},
    {"predicate": "knows_about        ", "include": True, "symmetric": False, "relationship_class": "knowledge"},
    {"predicate": "leader_of          ", "include": True, "symmetric": False, "relationship_class": "leadership"},
    {"predicate": "military_leader_of ", "include": True, "symmetric": False, "relationship_class": "leadership"},
    {"predicate": "neighbor_with      ", "include": True, "symmetric": True,  "relationship_class": "geopolitical"},
    {"predicate": "operates_at        ", "include": True, "symmetric": False, "relationship_class": "location"},
    {"predicate": "opposes            ", "include": True, "symmetric": True,  "relationship_class": "political"},
    {"predicate": "patrols_to         ", "include": True, "symmetric": False, "relationship_class": "activity"},
    {"predicate": "reports_to         ", "include": True, "symmetric": False, "relationship_class": "organizational"},
    {"predicate": "rescued            ", "include": True, "symmetric": False, "relationship_class": "event"},
    {"predicate": "spiritual_leader_of", "include": True, "symmetric": False, "relationship_class": "leadership"},
    {"predicate": "target_of          ", "include": True, "symmetric": False, "relationship_class": "event"},
    {"predicate": "tortured_by        ", "include": True, "symmetric": False, "relationship_class": "event"},
])
DF_PREDICATE_RULES["predicate"] = DF_PREDICATE_RULES["predicate"].str.strip()

In [66]:
# validation - find any predicates missing rule
display(
    DF_RELATIONSHIPS[["predicate"]]
    .drop_duplicates()
    .merge(
        DF_PREDICATE_RULES[["predicate"]],
        on="predicate",
        how="left",
        indicator=True,
    )
    .loc[lambda df: df["_merge"] == "left_only", ["predicate"]]
    .sort_values("predicate")
    .reset_index(drop=True)
)

,predicate


In [76]:
# Phase R8A: Export predicate semantic classes policy
# (generated from reviewed predicate rules)

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_PATH = (
    WORKING_DRAFTS_PATH
    / f"data_policy_predicate_semantic_classes_{timestamp}.csv"
)

(
    DF_PREDICATE_RULES
    .sort_values(["predicate"])
    .to_csv(OUTPUT_PATH, index=False)
)

print("Phase R8A OK: predicate semantic classes policy written.")
print(f"- Output path: {OUTPUT_PATH}")
print(f"- Rows written: {len(DF_PREDICATE_RULES)}")
print(f"- Columns: {len(DF_PREDICATE_RULES.columns)}")

print("\nNext step (human review required):")
print("1. Open the generated CSV file.")
print("2. Review each predicate and confirm the semantic policy values.")
print("3. Adjust any values if necessary.")
print("4. When satisfied, move the file to:")
print(f"   {INDEXES_RELPATH}/data_policy_predicate_semantic_classes.csv")
print("5. Remove the timestamp from the filename when promoting it.")

del timestamp, OUTPUT_PATH, datetime

Phase R8A OK: predicate semantic classes policy written.
- Output path: /Users/charissophia/obsidian/Iron Wolf Trading Company/_local/machine_wip/data_policy_predicate_semantic_classes_20260315_042915.csv
- Rows written: 24
- Columns: 4

Next step (human review required):
1. Open the generated CSV file.
2. Review each predicate and confirm the semantic policy values.
3. Adjust any values if necessary.
4. When satisfied, move the file to:
   _meta/indexes/data_policy_predicate_semantic_classes.csv
5. Remove the timestamp from the filename when promoting it.


## Phase R9: Normalize semantic relationships

This phase prepares the canonical relationship table for downstream semantic use.

It enriches each relationship with:

- `subject_type` derived from `subject_id`
- `object_type` derived from `object_id`
- predicate rule metadata from `DF_PREDICATE_RULES`

The result is a normalized semantic relationship dataframe that can be
used for graph export or semantic querying.

In [68]:
# Phase R9: Normalize semantic relationships
LAST_PHASE_RUN = "R9"

DF_RELATIONSHIP_SEMANTICS = (
    DF_RELATIONSHIPS
    .assign(
        subject_type=lambda df: df["subject_id"].str.split("_", n=1).str[0],
        object_type=lambda df: df["object_id"].str.split("_", n=1).str[0],
    )
    .merge(
        DF_PREDICATE_RULES,
        on="predicate",
        how="left",
    )
    .assign(
        pair_type=lambda df: df["subject_type"] + "|" + df["object_type"]
    )
    [
        [
            "subject_id",
            "subject_type",
            "predicate",
            "object_id",
            "object_type",
            "pair_type",
            "include",
            "symmetric",
            "relationship_class",
        ]
    ]
    .sort_values(
        ["predicate", "subject_id", "object_id"],
        ascending=[True, True, True],
    )
    .reset_index(drop=True)
)

print("Phase R9 OK: semantic relationships normalized.")
print(f"- Input relationships: {len(DF_RELATIONSHIPS):>8}")
print(f"- Output rows:         {len(DF_RELATIONSHIP_SEMANTICS):>8}")
print(f"- Distinct predicates: {DF_RELATIONSHIP_SEMANTICS['predicate'].nunique():>8}")
print(f"- Distinct pair types: {DF_RELATIONSHIP_SEMANTICS['pair_type'].nunique():>8}")

print("\nStructure:")
print(f"- rows: {len(DF_RELATIONSHIP_SEMANTICS)}")
print(f"- columns: {len(DF_RELATIONSHIP_SEMANTICS.columns)}")
print(f"- column names: {list(DF_RELATIONSHIP_SEMANTICS.columns)}")

print("\nSample rows:")
display(DF_RELATIONSHIP_SEMANTICS.head(5))

Phase R9 OK: semantic relationships normalized.
- Input relationships:      144
- Output rows:              144
- Distinct predicates:       24
- Distinct pair types:       12

Structure:
- rows: 144
- columns: 9
- column names: ['subject_id', 'subject_type', 'predicate', 'object_id', 'object_type', 'pair_type', 'include', 'symmetric', 'relationship_class']

Sample rows:


,subject_id,subject_type,predicate,object_id,object_type,pair_type,include,symmetric,relationship_class
0,faction_hands,faction,allied_with,faction_thieves_guild,faction,faction|faction,True,True,political
1,faction_michaeline,faction,allied_with,faction_remedy,faction,faction|faction,True,True,political
2,faction_remedy,faction,allied_with,faction_spencer,faction,faction|faction,True,True,political
3,faction_remedy,faction,allied_with,faction_tolanites,faction,faction|faction,True,True,political
4,faction_sleepsong,faction,allied_with,person_kir,person,faction|person,True,True,political


In [69]:
# Relationship Bootstrapping section cleanup
del VOCAB_ENTITIES_RELPATH
del VOCAB_RELATIONSHIPS_PATH
del VOCAB_RELATIONSHIPS_RELPATH

# Semantic Graph Preparation

This section derives graph-ready artifacts from the canonical
relationship layer.

Inputs to this section:

- DF_VOCAB_ENTITIES
- DF_RELATIONSHIPS
- DF_RELATIONSHIP_SEMANTICS
- DF_PREDICATE_RULES

The canonical relationship table is treated as authoritative world
knowledge. Phases in this section prepare that knowledge for semantic
graph construction and downstream analysis.

No source material or canonical vocabulary files are modified here.

## Phase G1: Build semantic relationship edges

This phase prepares and writes a graph-ready semantic edge table from the
normalized canonical relationship layer.

Only predicates marked for inclusion are retained.

The resulting artifact preserves semantic metadata needed for later
graph analysis and downstream query work.

In [79]:
# Phase G1: Write semantic relationship edges
LAST_PHASE_RUN = "G1"

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_PATH = (
    WORKING_DRAFTS_PATH
    / f"graph_semantic_edges_{INDEX_VERSION.lower()}_{timestamp}.csv"
)

(
    DF_RELATIONSHIP_SEMANTICS
    .loc[lambda df: df["include"]]
    [
        [
            "subject_id",
            "predicate",
            "object_id",
            "subject_type",
            "object_type",
            "pair_type",
            "symmetric",
            "relationship_class",
        ]
    ]
    .sort_values(
        ["predicate", "subject_id", "object_id"],
        ascending=[True, True, True],
    )
    .to_csv(OUTPUT_PATH, index=False)
)

print("Phase G1 OK: semantic relationship edges written.")
print(f"- Output path: {OUTPUT_PATH}")
print(f"- Rows written: {len(DF_RELATIONSHIP_SEMANTICS.loc[DF_RELATIONSHIP_SEMANTICS['include']]):>8}")

print("\nNext step (human review required):")
print("1. Open the generated CSV file.")
print("2. Confirm the semantic edge rows look correct.")
print("3. When satisfied, move the file to:")
print(f"   {INDEXES_RELPATH}/graph_relationship_edges_{INDEX_VERSION.lower()}.csv")
print("4. Remove the timestamp from the filename when promoting it.")

del timestamp, OUTPUT_PATH, datetime

Phase G1 OK: semantic relationship edges written.
- Output path: /Users/charissophia/obsidian/Iron Wolf Trading Company/_local/machine_wip/graph_semantic_edges_v0_20260315_050109.csv
- Rows written:      144

Next step (human review required):
1. Open the generated CSV file.
2. Confirm the semantic edge rows look correct.
3. When satisfied, move the file to:
   _meta/indexes/graph_relationship_edges_v0.csv
4. Remove the timestamp from the filename when promoting it.


## Phase G2: Build semantic graph nodes

This phase builds and writes a graph-ready node table for the semantic
graph layer.

Nodes are derived from the distinct relationship endpoints referenced by
included semantic edges and are labeled from the canonical entity vocabulary.

The output mirrors the existing graph node schema:

- `node_id`
- `node_type`
- `label`

In [80]:
# Phase G2: Export semantic graph nodes
LAST_PHASE_RUN = "G2"

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_PATH = (
    WORKING_DRAFTS_PATH
    / f"graph_semantic_nodes_{INDEX_VERSION.lower()}_{timestamp}.csv"
)

(
    pd.concat(
        [
            DF_RELATIONSHIP_SEMANTICS.loc[lambda df: df["include"], "subject_id"],
            DF_RELATIONSHIP_SEMANTICS.loc[lambda df: df["include"], "object_id"],
        ],
        ignore_index=True,
    )
    .drop_duplicates()
    .to_frame(name="node_id")
    .assign(
        node_type=lambda df: df["node_id"].str.split("_", n=1).str[0]
    )
    .merge(
        DF_VOCAB_ENTITIES[["entity_id", "canonical_name"]],
        left_on="node_id",
        right_on="entity_id",
        how="left",
    )
    .drop(columns="entity_id")
    .rename(columns={"canonical_name": "label"})
    .sort_values(["node_type", "node_id"], ascending=[True, True])
    .reset_index(drop=True)
    .to_csv(OUTPUT_PATH, index=False)
)

print("Phase G2 OK: semantic graph nodes written.")
print(f"- Output path: {OUTPUT_PATH}")
print(
    f"- Rows written: {pd.concat([DF_RELATIONSHIP_SEMANTICS.loc[lambda df: df['include'], 'subject_id'], DF_RELATIONSHIP_SEMANTICS.loc[lambda df: df['include'], 'object_id']], ignore_index=True).drop_duplicates().shape[0]}"
)
print("- Columns: 3")

print("\nNext step (human review required):")
print("1. Open the generated CSV file.")
print("2. Confirm node_id, node_type, and label look correct.")
print("3. When satisfied, move the file to:")
print(f"   {INDEXES_RELPATH}/graph_semantic_nodes_{INDEX_VERSION.lower()}.csv")
print("4. Remove the timestamp from the filename when promoting it.")

del timestamp, OUTPUT_PATH, datetime

Phase G2 OK: semantic graph nodes written.
- Output path: /Users/charissophia/obsidian/Iron Wolf Trading Company/_local/machine_wip/graph_semantic_nodes_v0_20260315_051336.csv
- Rows written: 108
- Columns: 3

Next step (human review required):
1. Open the generated CSV file.
2. Confirm node_id, node_type, and label look correct.
3. When satisfied, move the file to:
   _meta/indexes/graph_semantic_nodes_v0.csv
4. Remove the timestamp from the filename when promoting it.


## Phase G3: Audit semantic graph artifacts

This phase performs a small integrity audit of the semantic graph layer.

Checks performed:

- every semantic edge endpoint exists in the semantic node table
- semantic node IDs are unique
- profiles node counts by type

No files are written in this phase.

In [84]:
# Phase G3: Audit semantic graph artifacts
LAST_PHASE_RUN = "G3"

DF_GRAPH_SEMANTIC_NODES = pd.read_csv(
    INDEXES_PATH / f"graph_semantic_nodes_{INDEX_VERSION.lower()}.csv",
    dtype=str,
).fillna("")

DF_GRAPH_SEMANTIC_EDGES = pd.read_csv(
    INDEXES_PATH / f"graph_semantic_edges_{INDEX_VERSION.lower()}.csv",
    dtype=str,
).fillna("")

NODE_IDS = set(DF_GRAPH_SEMANTIC_NODES["node_id"])

missing_subjects = (
    DF_GRAPH_SEMANTIC_EDGES
    .loc[~DF_GRAPH_SEMANTIC_EDGES["subject_id"].isin(NODE_IDS), "subject_id"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

missing_objects = (
    DF_GRAPH_SEMANTIC_EDGES
    .loc[~DF_GRAPH_SEMANTIC_EDGES["object_id"].isin(NODE_IDS), "object_id"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

duplicate_nodes = (
    DF_GRAPH_SEMANTIC_NODES["node_id"]
    .value_counts()
    .loc[lambda s: s > 1]
    .rename_axis("node_id")
    .reset_index(name="duplicate_count")
)

if not missing_subjects.empty:
    print("Missing subject_ids from semantic nodes:")
    display(missing_subjects)

if not missing_objects.empty:
    print("Missing object_ids from semantic nodes:")
    display(missing_objects)

if not duplicate_nodes.empty:
    print("Duplicate node_ids in semantic nodes:")
    display(duplicate_nodes)

if not missing_subjects.empty or not missing_objects.empty or not duplicate_nodes.empty:
    raise ValueError(
        "Semantic graph audit failed.\n\n"
        "What to do:\n"
        "- Regenerate the semantic graph artifacts\n"
        "- Confirm graph_semantic_edges_v0.csv and graph_semantic_nodes_v0.csv were promoted together\n"
        "- Rerun this phase"
    )

print("Phase G3 OK: semantic graph artifacts are internally consistent.")
print(f"- Semantic nodes: {len(DF_GRAPH_SEMANTIC_NODES):>8}")
print(f"- Semantic edges: {len(DF_GRAPH_SEMANTIC_EDGES):>8}")
print(f"- Distinct node types: {DF_GRAPH_SEMANTIC_NODES['node_type'].nunique():>8}")

print("\nNode counts by type:")
display(
    DF_GRAPH_SEMANTIC_NODES["node_type"]
    .value_counts()
    .rename_axis("node_type")
    .reset_index(name="node_count")
)

del DF_GRAPH_SEMANTIC_NODES, DF_GRAPH_SEMANTIC_EDGES
del NODE_IDS, missing_subjects, missing_objects, duplicate_nodes

Phase G3 OK: semantic graph artifacts are internally consistent.
- Semantic nodes:      108
- Semantic edges:      144
- Distinct node types:        6

Node counts by type:


,node_type,node_count
0,person,60
1,place,15
2,faction,14
3,org,10
4,creat,6
5,artifact,3
